In [44]:
import numpy as np
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder

In [45]:
# Load MNIST dataset
mnist = fetch_openml('mnist_784', version=1)

X = mnist.data.values / 255.0
y = mnist.target.astype(int).values.reshape(-1,1)

encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

In [46]:
# Activation functions

def relu(x):
    return np.maximum(0, x)

def relu_derivative(x):
    return x > 0

def softmax(x):
    exp_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return exp_x / np.sum(exp_x, axis=1, keepdims=True)

In [49]:
# Weight initialization (He initialization for ReLU layers, Xavier for output)

W1 = np.random.randn(input_size, hidden1) * np.sqrt(2. / input_size)
b1 = np.zeros((1, hidden1))

W2 = np.random.randn(hidden1, hidden2) * np.sqrt(2. / hidden1)
b2 = np.zeros((1, hidden2))

W3 = np.random.randn(hidden2, output_size) * np.sqrt(1. / hidden2) # Xavier initialization
b3 = np.zeros((1, output_size))

learning_rate = 0.001 # Reduced learning rate
epochs = 50 # Increased epochs
batch_size = 64 # Mini-batch size

num_batches = len(X_train) // batch_size

In [50]:
for epoch in range(epochs):
    # Shuffle training data for each epoch
    permutation = np.random.permutation(len(X_train))
    X_train_shuffled = X_train[permutation]
    y_train_shuffled = y_train[permutation]

    epoch_loss = 0

    for i in range(num_batches):
        start = i * batch_size
        end = start + batch_size

        X_batch = X_train_shuffled[start:end]
        y_batch = y_train_shuffled[start:end]

        # Forward propagation
        Z1 = np.dot(X_batch, W1) + b1
        A1 = relu(Z1)

        Z2 = np.dot(A1, W2) + b2
        A2 = relu(Z2)

        Z3 = np.dot(A2, W3) + b3
        A3 = softmax(Z3)

        # Cross entropy loss
        loss = -np.mean(y_batch * np.log(A3 + 1e-8))
        epoch_loss += loss

        # Backpropagation
        dZ3 = A3 - y_batch
        dW3 = np.dot(A2.T, dZ3)
        db3 = np.sum(dZ3, axis=0, keepdims=True)

        dA2 = np.dot(dZ3, W3.T)
        dZ2 = dA2 * relu_derivative(Z2)
        dW2 = np.dot(A1.T, dZ2)
        db2 = np.sum(dZ2, axis=0, keepdims=True)

        dA1 = np.dot(dZ2, W2.T)
        dZ1 = dA1 * relu_derivative(Z1)
        dW1 = np.dot(X_batch.T, dZ1)
        db1 = np.sum(dZ1, axis=0, keepdims=True)

        # Update weights
        W3 -= learning_rate * dW3
        b3 -= learning_rate * db3

        W2 -= learning_rate * dW2
        b2 -= learning_rate * db2

        W1 -= learning_rate * dW1
        b1 -= learning_rate * db1

    print(f"Epoch {epoch+1}, Loss: {epoch_loss/num_batches}")

Epoch 1, Loss: 0.043372456782731814
Epoch 2, Loss: 0.019387613278741216
Epoch 3, Loss: 0.014091393931224385
Epoch 4, Loss: 0.011184391022201192
Epoch 5, Loss: 0.009161649564530295
Epoch 6, Loss: 0.007791176478907487
Epoch 7, Loss: 0.00658720016013764
Epoch 8, Loss: 0.0057430778321263315
Epoch 9, Loss: 0.005005344451563517
Epoch 10, Loss: 0.004439773817669988
Epoch 11, Loss: 0.003962355263390787
Epoch 12, Loss: 0.003392714962405501
Epoch 13, Loss: 0.003100242335186007
Epoch 14, Loss: 0.002683774672447616
Epoch 15, Loss: 0.00246356767991857
Epoch 16, Loss: 0.002167217199888826
Epoch 17, Loss: 0.0018874139873601512
Epoch 18, Loss: 0.0016440348071615112
Epoch 19, Loss: 0.0014993829666158547
Epoch 20, Loss: 0.0012451290550846593
Epoch 21, Loss: 0.0011301569103904449
Epoch 22, Loss: 0.001010281963340396
Epoch 23, Loss: 0.0008905223484495869
Epoch 24, Loss: 0.0007449536080657431
Epoch 25, Loss: 0.0006961485460343541
Epoch 26, Loss: 0.0006441732950413887
Epoch 27, Loss: 0.0005653246352946014
E

In [51]:
# Testing

Z1 = np.dot(X_test, W1) + b1
A1 = relu(Z1)

Z2 = np.dot(A1, W2) + b2
A2 = relu(Z2)

Z3 = np.dot(A2, W3) + b3
A3 = softmax(Z3)

predictions = np.argmax(A3, axis=1)
true_labels = np.argmax(y_test, axis=1)

accuracy = np.mean(predictions == true_labels)

print("Test Accuracy:", accuracy)

Test Accuracy: 0.9764285714285714
